# 📘 Tutorial: Understanding and Visualizing Raw COLMAP (SfM) Outputs

This tutorial is designed to explain the core data structures produced by **COLMAP (Structure-from-Motion)** and directly visualize each component in raw form without downstream filtering or semantic processing.

---

### 📑 Table of Contents:
1. **Overview of Core COLMAP Text Files** (`cameras.txt`, `images.txt`, `points3D.txt`)
2. **Exploring `cameras.txt`**: Camera Intrinsics Matrix $K$, Focal Length $f$, Principal Point $(c_x, c_y)$, and Radial Distortion $k_1$
3. **Exploring `images.txt`**: Camera Extrinsics, Quaternion $\to$ World Camera Center $C = -R^T T$
4. **Exploring 2D Feature Tracks**: Visualizing $(u, v)$ keypoints and distinguishing matched vs. unmatched points
5. **Exploring `points3D.txt`**: Spatial Coordinates $(x, y, z)$, Raw RGB Colors, and Reprojection Error
6. **Full 3D Scene Visualization**: Raw Point Cloud + Camera Frustums + Multi-View Triangulation Rays

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation

# Configure plotting aesthetics
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("✅ Python environment ready!")

## 1. Core Concept: What Does COLMAP Output?

After running Structure-from-Motion (SfM) on multi-view UAV images, COLMAP produces **3 fundamental data files**:

| File | Concept | Physical / Mathematical Meaning |
| :--- | :--- | :--- |
| `cameras.txt` | **Camera Intrinsics** | Internal optical parameters of the camera lens (focal length, principal point, radial lens distortion). |
| `images.txt` | **Camera Extrinsics & 2D Observations** | 6-DOF camera poses of each drone frame + 2D feature keypoints $(u, v)$ with 3D point track IDs. |
| `points3D.txt` | **3D Sparse Points & Tracks** | 3D world coordinates $(x, y, z)$, original RGB colors, reprojection error, and list of observing image frames. |

Let us explore and visualize each file in detail below:

## 2. Exploring `cameras.txt` (Camera Intrinsics)

Each line in `cameras.txt` describes a camera calibration:
```text
# CAMERA_ID  MODEL          WIDTH  HEIGHT  PARAMS[f, cx, cy, k1]
  1          SIMPLE_RADIAL  1320   989     925.7016 660.0 494.5 0.0124
```

- **$f = 925.7$ px**: Focal length in pixels.
- **$c_x = 660.0, c_y = 494.5$ px**: Principal point (center of projection, $1320/2, 989/2$).
- **$k_1$**: Radial distortion coefficient ($k_1 > 0$ for barrel distortion, $k_1 < 0$ for pincushion distortion).

In [ ]:
# Simulated camera intrinsics from cameras.txt
camera_model = {
    "id": 1,
    "model": "SIMPLE_RADIAL",
    "width": 1320,
    "height": 989,
    "f": 925.7016,
    "cx": 660.0,
    "cy": 494.5,
    "k1": 0.015  # Sample radial distortion coefficient
}

# Camera Intrinsics Matrix K (3x3)
K = np.array([
    [camera_model["f"], 0, camera_model["cx"]],
    [0, camera_model["f"], camera_model["cy"]],
    [0, 0, 1]
])

print("📐 Camera Intrinsics Matrix K (3x3):")
print(K)

# Visualize the radial lens distortion effect on pixel grid
gx, gy = np.meshgrid(np.linspace(0, camera_model["width"], 25), np.linspace(0, camera_model["height"], 20))
u_norm = (gx - camera_model["cx"]) / camera_model["f"]
v_norm = (gy - camera_model["cy"]) / camera_model["f"]
r2 = u_norm**2 + v_norm**2
u_dist = gx + (gx - camera_model["cx"]) * camera_model["k1"] * r2
v_dist = gy + (gy - camera_model["cy"]) * camera_model["k1"] * r2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(gx, gy, c='blue', s=8)
ax1.set_title("Ideal Pixel Grid (Undistorted / Linear Pin-hole)")
ax1.set_xlim(0, camera_model["width"]); ax1.set_ylim(camera_model["height"], 0)
ax1.set_aspect('equal'); ax1.grid(True, linestyle=':', alpha=0.5)

ax2.scatter(u_dist, v_dist, c='red', s=8)
ax2.set_title(f"Distorted Pixel Grid (Radial Distortion k1 = {camera_model['k1']})")
ax2.set_xlim(0, camera_model["width"]); ax2.set_ylim(camera_model["height"], 0)
ax2.set_aspect('equal'); ax2.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Exploring `images.txt` (Camera Extrinsics & 2D Observations)

In `images.txt`, every registered image occupies **2 consecutive lines**:

- **Line 1 (Extrinsic Pose)**: `IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME`
  - $[q_w, q_x, q_y, q_z]$: Unit quaternion representing camera rotation matrix $R$.
  - $[t_x, t_y, t_z]$: Translation vector $T$ from world space to camera space.
  - ⚠️ **Key Note**: The real 3D physical position of the UAV is: **$C = -R^T \cdot T$**.

- **Line 2 (2D Feature Observations)**: Sequence of triplets `X_pixel, Y_pixel, POINT3D_ID`
  - `POINT3D_ID = -1`: 2D keypoint detected but unmatched with other views (no 3D point).
  - `POINT3D_ID >= 0`: 2D keypoint **successfully triangulated** into a 3D point with the given ID.

In [ ]:
# Convert Quaternion and Translation vector to World Camera Center C
sample_image_record = {
    "image_id": 1,
    "name": "001.png",
    "qvec": np.array([0.8535, -0.1464, 0.3535, 0.3535]), # [qw, qx, qy, qz]
    "tvec": np.array([-12.4, 4.8, 45.2])                 # [tx, ty, tz]
}

# 1. Convert Quaternion -> 3x3 Rotation Matrix R
r_quat = sample_image_record["qvec"]
rot_matrix = Rotation.from_quat([r_quat[1], r_quat[2], r_quat[3], r_quat[0]]).as_matrix()

# 2. Compute Physical World Camera Center C = -R^T * T
camera_center_world = -rot_matrix.T @ sample_image_record["tvec"]

print(f"📷 Image: {sample_image_record['name']} (ID: {sample_image_record['image_id']})")
print(f"   • 3x3 Rotation Matrix R:\n{rot_matrix}")
print(f"   • Translation Vector T (World -> Cam): {sample_image_record['tvec']}")
print(f"   📍 True UAV Position in World Coordinates C = -R^T * T: {camera_center_world.round(2)} meters")

## 4. Visualizing 2D Feature Keypoints: Matched vs. Unmatched Points

Here we plot all 2D feature keypoints extracted on an image to show the difference between points that formed 3D correspondences (`POINT3D_ID >= 0`) versus unmatched keypoints (`POINT3D_ID = -1`).

In [ ]:
np.random.seed(101)
n_features = 300

# Simulated 2D keypoint coordinates on image canvas
u_pts = np.random.uniform(50, camera_model["width"] - 50, n_features)
v_pts = np.random.uniform(50, camera_model["height"] - 50, n_features)

# 70% matched into 3D (point3D_id >= 0), 30% unmatched outliers (point3D_id = -1)
point3d_ids = np.array([i if np.random.rand() > 0.3 else -1 for i in range(n_features)])

matched_mask = point3d_ids >= 0
unmatched_mask = point3d_ids == -1

plt.figure(figsize=(13, 7))
# Simulated dark canvas representing UAV frame
plt.imshow(np.ones((camera_model["height"], camera_model["width"], 3), dtype=np.uint8) * 40)

# Plot unmatched points (grey)
plt.scatter(u_pts[unmatched_mask], v_pts[unmatched_mask], c='gray', s=20, alpha=0.5, 
            label=f"Unmatched 2D Keypoints (ID = -1): {np.sum(unmatched_mask)} pts")

# Plot successfully triangulated 3D points (bright green)
plt.scatter(u_pts[matched_mask], v_pts[matched_mask], c='lime', s=45, edgecolors='black', linewidth=0.5,
            label=f"Triangulated 3D Points (ID >= 0): {np.sum(matched_mask)} pts")

plt.title(f"Visualization of images.txt Line 2: 2D Keypoints Distribution on Frame {sample_image_record['name']}", fontsize=12)
plt.xlabel("Pixel U (px)"); plt.ylabel("Pixel V (px)")
plt.xlim(0, camera_model["width"]); plt.ylim(camera_model["height"], 0) # Top-left origin
plt.legend(loc='upper right', framealpha=0.9)
plt.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Exploring `points3D.txt` (3D Points, Error, and Track Length)

Each line in `points3D.txt` represents one triangulated 3D point:
```text
# POINT3D_ID  X       Y       Z       R    G    B    ERROR  TRACK[IMAGE_ID, POINT2D_IDX, ...]
  1402        -15.42  3.18    120.45  180  175  160  0.48   1 42 5 18 12 99
```

### 2 Most Critical Quality Indicators:
1. **`ERROR` (Reprojection Error)**: Distance in pixels between observed 2D keypoints and projected 3D point ($< 1.0\text{ px}$ indicates high geometric precision).
2. **`TRACK LENGTH`**: Number of camera views observing this 3D point (e.g. `1 42 5 18 12 99` $\to$ observed by images 1, 5, 12). Longer tracks indicate higher spatial certainty.

In [ ]:
# Statistical distribution analysis of raw COLMAP 3D point cloud
np.random.seed(42)
n_pts_demo = 5000

# Simulated reprojection error distribution (Gamma distribution peaking around 0.4 - 0.7 px)
reproj_errors = np.random.gamma(shape=2.5, scale=0.25, size=n_pts_demo)

# Simulated track length distribution (number of camera views per point)
track_lengths = np.random.geometric(p=0.2, size=n_pts_demo) + 1
track_lengths = np.clip(track_lengths, 2, 35)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram 1: Reprojection Error
ax1.hist(reproj_errors, bins=40, color='#e74c3c', edgecolor='black', alpha=0.75)
ax1.axvline(np.mean(reproj_errors), color='blue', linestyle='--', linewidth=2, label=f"Mean Error: {np.mean(reproj_errors):.2f} px")
ax1.axvline(1.0, color='black', linestyle=':', linewidth=2, label="Standard Threshold (1.0 px)")
ax1.set_title("Reprojection Error Distribution (pixels)")
ax1.set_xlabel("Error (pixels)"); ax1.set_ylabel("Number of 3D Points")
ax1.legend(); ax1.grid(True, linestyle=':', alpha=0.5)

# Histogram 2: Track Length
ax2.hist(track_lengths, bins=range(2, 35), color='#3498db', edgecolor='black', alpha=0.75)
ax2.axvline(np.mean(track_lengths), color='red', linestyle='--', linewidth=2, label=f"Mean Views: {np.mean(track_lengths):.1f} cameras")
ax2.set_title("Track Length Distribution (Multi-View Coverage)")
ax2.set_xlabel("Number of Observing Cameras per 3D Point"); ax2.set_ylabel("Number of 3D Points")
ax2.legend(); ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Full 3D Scene Visualization: Point Cloud + Camera Poses + Triangulation Rays

Here we reconstruct the complete 3D geometric scene from raw COLMAP outputs:
1. **3D Point Cloud**: Displayed with original RGB texture colors.
2. **UAV Camera Poses**: Flight trajectory with camera positions in 3D world coordinates.
3. **Multi-View Triangulation Rays**: Optical lines of sight from multiple camera centers converging at a single 3D point.

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax = fig.add_subplot(111, projection='3d')

# 1. Generate 3D cable-stayed bridge geometry with raw RGB colors
np.random.seed(42)
n_deck = 800
x_deck = np.linspace(-40, 40, n_deck)
y_deck = np.zeros(n_deck) + np.random.normal(0, 0.1, n_deck)
z_deck = np.random.uniform(-3, 3, n_deck)
rgb_deck = np.tile([0.6, 0.6, 0.65], (n_deck, 1)) # Concrete grey

# Tower pylons
x_tower = np.concatenate([np.repeat(-15, 200), np.repeat(15, 200)]) + np.random.normal(0, 0.2, 400)
y_tower = np.concatenate([np.linspace(0, 25, 200), np.linspace(0, 25, 200)])
z_tower = np.zeros(400) + np.random.normal(0, 0.3, 400)
rgb_tower = np.tile([0.8, 0.8, 0.85], (400, 1))

all_x = np.concatenate([x_deck, x_tower])
all_y = np.concatenate([y_deck, y_tower])
all_z = np.concatenate([z_deck, z_tower])
all_rgb = np.vstack([rgb_deck, rgb_tower])

# Plot raw 3D point cloud
ax.scatter(all_x, all_z, all_y, c=all_rgb, s=3.0, alpha=0.7, label="Raw 3D Point Cloud (RGB)")

# 2. Plot 12 UAV camera poses along flight trajectory
t_vals = np.linspace(-45, 45, 12)
cam_pos = np.array([[t, 10.0 + 4.0*np.sin(t/10), 16.0] for t in t_vals])
target_point = np.array([0.0, 0.0, 12.0]) # Target 3D point on tower top

# Plot drone camera positions and path
ax.scatter(cam_pos[:, 0], cam_pos[:, 2], cam_pos[:, 1], c='red', s=50, marker='^', label="UAV Camera Positions")
ax.plot(cam_pos[:, 0], cam_pos[:, 2], cam_pos[:, 1], 'r--', alpha=0.5, label="Drone Flight Path")

# 3. Draw multi-view triangulation rays converging at target 3D point
active_cams = [3, 5, 7, 9]
for idx in active_cams:
    cp = cam_pos[idx]
    ax.plot([cp[0], target_point[0]], [cp[2], target_point[2]], [cp[1], target_point[1]], 
            color='cyan', linewidth=1.8, linestyle='-')

# Highlight triangulated 3D intersection point
ax.scatter([target_point[0]], [target_point[2]], [target_point[1]], color='yellow', s=120, 
           edgecolors='black', linewidth=2, label="Triangulated 3D Point (Ray Intersection)", zorder=10)

ax.set_title("Visualization of Raw COLMAP Structure: Camera Poses & Multi-View 3D Triangulation", fontsize=13, pad=15)
ax.set_xlabel("Longitudinal Axis X (m)")
ax.set_ylabel("Lateral Axis Z (m)")
ax.set_zlabel("Elevation Axis Y (m)")
ax.legend(loc='upper right', framealpha=0.9)
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

---
## 🎯 Key Takeaways on COLMAP Outputs:

1. **COLMAP is Purely Geometric**: It solves perspective equations to determine camera positions and 3D point coordinates $(x, y, z)$. It has **no concept of semantic class labels**.
2. **The Crucial Feature Track Link**: The `TRACK` list in `points3D.txt` (and `POINT3D_ID` in `images.txt`) links every 3D coordinate to its exact 2D pixel coordinates across camera views.
3. **Bridging to Deep Learning**: By querying 2D deep learning segmentation masks at these exact pixel coordinates and aggregating votes across views, the raw geometric cloud is transformed into a high-fidelity **Semantic 3D Digital Twin**.